# Explore here

In [ ]:
#LIBRERÍAS A UTILIZAR

import pandas as pd


: 

In [ ]:
df = pd.read_csv('https://storage.googleapis.com/breathecode/project-files/bank-marketing-campaign-data.csv', sep=';')
df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
categoricas = df.select_dtypes(include=['str']).columns.tolist() # Esta función sirve para obtener las columnas de algún tipo

categoricas

# Y así obtenemos las que son de tipo 'str'

In [ ]:
# Y para obtener las numéricas es un proceso casi igual:

numericas = df.select_dtypes(exclude=['str']).columns.tolist()

numericas

In [ ]:
# Antes de empezar a tocar estas columnas y empezar a trabajar en ellas, lo primero que tenemos que hacer es ver si hay duplicados:

df.duplicated().sum()

In [ ]:
# si tenemos filas duplicadas entonces:

if df.duplicated().sum():
    df.drop_duplicates(inplace= True)

df.shape

# Con esto, quitamos las 12 duplicadas.

In [ ]:
# Vamos a hacer un FOR para transformar variables categóricas en numéricas:

for columna in categoricas:
    df[columna] = pd.factorize(df[columna])[0]

df.info()

# Y todas se transforman en numéricas.

In [ ]:
df.head()

# El NO quedó como 0, y el SÍ quedó como 1. Tenemos 20 predictoras, es un modelo grande en sí, pero no necesariamente todas las columnas van a servir, y eso lo vemos con un EDA. 

# AQUÍ VA EL EDA (HACERLO SOLITOS)

El fin es crear un algoritmo de clasificación que ayude a **predecir si un cliente contratará o no un depósito a largo plazo**.

Variable objetivo: 'y'


### 1) Eliminar información irrelevante:

**Debemos responder a la siguiente pregunta: ¿son todas las características imprescindibles para realizar una predicción? Normalmente, esa pregunta es un rotundo no.**

En este caso en particular, considero que todas las variables son dignas de analizarse, porque todas están relacionadas con el problema. No existen índices ni nombres, ni ID, que no tengan que ver con un préstamo o con el contacto del cliente, por ende no descartaremos ninguna.

## 2. Eliminamos Duplicados: 
A este paso lo realizamos con anterioridad. 

## 3. Análisis de variables univariante

Esto es el análisis columna a columna del DataFrame. Para ello, debemos distinguir si una variable es categórica o numérica, ya que el análisis y las conclusiones que se pueden obtener serán distintas.

In [ ]:
numericas = [ 'age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'
]

categoricas = [ 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome'
]   


Análisis de variables CATEGÓRICAS:

Para representar este tipo de variables utilizaremos histogramas o gráficos de barras verticales.

In [ ]:
import math
import matplotlib.pyplot as plt
import seaborn as sns

# Lista con tus 10 variables categóricas
categoricas = [
    'job', 'marital', 'education', 'default', 'housing', 
    'loan', 'contact', 'month', 'day_of_week', 'poutcome'
]

# Configuración de la retícula: 3 columnas
ncols = 3
num_vars = len(categoricas)
nrows = math.ceil(num_vars / ncols)  # Dará 4 filas (4x3 = 12 espacios)

fig, axis = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 14))

# Bucle para graficar automáticamente cada variable de la lista
for i, col in enumerate(categoricas):
    row = i // ncols
    c = i % ncols
    ax = axis[row, c]
    
    # Crear histograma para cada variable
    sns.histplot(ax=ax, data=total_data, x=col)
    ax.set_title(f'Distribución de {col}', fontsize=10, fontweight='bold')
    
    # Rotar las etiquetas del eje X si tienen nombres largos para evitar solapamiento
    ax.tick_params(axis='x', rotation=30)
    
    # Opcional: eliminar la etiqueta repetitiva de 'Count' salvo en la primera columna
    if c != 0:
        ax.set(ylabel=None)

# Ocultar los subgráficos sobrantes (las celdas 11 y 12 que quedan vacías)
for j in range(num_vars, nrows * ncols):
    row = j // ncols
    c = j % ncols
    fig.delaxes(axis[row, c])

plt.tight_layout()
plt.show()

Análisis de variables NUMÉRICAS:

## 4. Análisis de variables multivariante. Análisis numérico-numérico.
Tras analizar las características una a una, es momento de analizarlas en relación con la predictora y con ellas mismas, para sacar conclusiones más claras acerca de sus relaciones y poder tomar decisiones sobre su procesamiento.

Para comparar dos columnas numéricas se utilizan diagramas de dispersión y análisis de correlaciones.

Pero antes vamos a realizar una Matríz de correlación, ya que esta es una tabla que muestra los coeficientes de correlación entre varias variables numéricas al mismo tiempo. 
Cada celda de la tabla te indica qué tan fuertemente se relacionan dos variables entre sí:

1.0: Correlación positiva perfecta (si una sube, la otra también sube en la misma proporción).

0.0: Sin relación lineal.

-1.0: Correlación negativa perfecta (si una sube, la otra baja).

In [ ]:
# Antes del modelo, tengo que definir mis x de train y de test:

x = df.drop(columns=['y'])
y= df['y']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42 )

In [ ]:
# Tenemos que hacer la normalización de los datos (chequear si usamos el "standar" si sé que mis datos distribuyen normal, si no lo sé, voy a usar minmax) Asique en este caso lo vamos a hacer con minmax porque todavía no hacemos el Eda y estamos como ciegos. Luego que de que hagan el EDa pueden tomar una mejor decisión.

from sklearn.preprocessing import MinMaxScaler

scaler= MinMaxScaler()

x_train_esc = scaler.fit_transform(x_train)

x_train_esc = pd.DataFrame(x_train_esc, columns= x_train.columns, index= x_train.index)

x_test_esc = scaler.transform(x_test)

x_test_esc = pd.DataFrame(x_test_esc, columns =x_test.columns, index= x_test.index)




In [ ]:
x_train_esc.head() # para que los podamos ver. Y ahí ya están escalados los datos.

In [ ]:
# Una vez que todas las columnas son numéricas y están escaladas, podemos pensar en hacer una REGRESIÓN LOGÍSTICA:

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(x_train_esc, y_train)

# Y tenemos todos los hiperparámetros que está utilizando el modelo por defecto. Y tmb podemos ver los atributos del modelo, que es lo que usó para hacer las predicciones.

In [ ]:
# Hasta acá ya tengo un modelo entrenado sobre mis x de entrenamiento (x_train). 
# Ahora lo que necesito son resultados. 

y_pred = model.predict(x_test_esc)
y_pred

# ¿Cómo se va an a ver? En este caso es un Array de 8.236 de largo, que es el 20% + o - de los 41 mil casos que tengo, y voy a ver un conjunto de unos y ceros. Esto sólo me dice que ya el modelo hizo predicciones. 

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

# Acá comparamos los y de test con los y predichos. 
# y nos dice que este modelo (sin el EDA realizado todavía) tiene un 90.23 % de probabilidad de acertar. Si yo ahora analizo un nuevo cliente, mi modelo va a ser capaz de predecir en un 90% si ese cliente va a tomar o no va a tomar un crédito. 

In [ ]:
# Matríz de confusión

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
disp= ConfusionMatrixDisplay (cm, display_labels=['No toma deposito', 'Si toma deposito'])

disp.plot(cmap=plt.cm.Blues)

plt.xlabel('Predicciones del modelo')
plt.ylabel('Valores reales')
plt.show()


# En el EDA tenemos que mejorar todavía más el modelo.

In [ ]:
# Una vez que ya tengo el modelo base, ya podemos hacer la búsqueda de hiperparámetros

from sklearn.model_selection import GridSearchCV
import numpy as np

param_grid = {
    'penalty' : ['l1', 'l2', 'elasticnet', None],
    'C': [0.001, 0.01, 0.1, 10.0, 100.0, 1000.0],
    'solver': ['saga'],
    'l1_ratio': [0.5],
    'max_iter': [2000]
}

grid_search = GridSearchCV (
    estimator = model,
    param_grid= param_grid,
    cv = 5,
    scoring ='accuracy',
    n_jobs= -1
)


In [ ]:
# Ahí está definida la grilla, ahora hay que entrenarla

grid_search.fit(x_train_esc, y_train)


In [ ]:
print(grid_search.best_params_)

In [ ]:
mejor_modelo = grid_search.best_estimator_

y_pred_mejor = mejor_modelo.predict(x_test_esc)
accuracy_score(y_test, y_pred_mejor)

# No es una gran mejora la que obtuvimos, pero mejoró a 90.38 %. 